# Fairness analysis

Evaluates the saved validation predictions from the neural network, LightGBM, and CatBoost notebooks for **gender** and **age group**, without refitting any model. It reads the applicant-level files in `results/predictions/` that section 7 of each model notebook exports.

**How to read the outcomes.** A *positive prediction* means "predicted to default", which is the adverse outcome (denial or extra scrutiny). So:

| Quantity | Meaning for an applicant group |
|---|---|
| Selection (denial) rate | share of the group predicted to default |
| **FPR** | share of *creditworthy* applicants wrongly flagged. This is the main harm to applicants, and it is the "equal opportunity" gap in lending terms |
| **TPR** | share of *actual defaulters* correctly flagged. Gaps here mean the bank loses money unevenly |
| PPV | of those flagged, the share who really default (predictive parity) |
| Calibration gap | mean predicted probability minus observed default rate within the group |
| AUC | how well the model ranks risk *within* the group |

**Operating points.** Models are trained on roughly 8% positives, so a fixed 0.5 cut-off flags very few applicants and most rate-based fairness gaps collapse toward zero for the wrong reason. The notebook therefore reports two operating points: the pipeline's fixed 0.5 threshold, and a *prevalence-matched* threshold that flags as many applicants as actually default. Section 6 flags runs where the fixed threshold is degenerate.

**Metric choice for model selection** (assignment: one optimizing metric, optional satisficing metrics). The optimizing metric stays validation ROC-AUC. Fairness enters as satisficing filters (FPR gap, TPR gap, denial-rate ratio) with tolerances you set in the config cell.

**Contents**
1. Setup and configuration
2. Load predictions and choose runs
3. Group composition and base rates
4. Per-group performance (focus model)
5. Plots: rates, ROC, calibration
6. Fairness gaps across all runs
7. Uncertainty (bootstrap)
8. Intersectional view
9. Fairness vs. performance trade-off and shortlist
10. Mitigation experiment (group-specific thresholds)
11. Export and auto-generated findings
12. Ethical considerations and limitations

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import os, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import rankdata
from sklearn.metrics import roc_curve, average_precision_score

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
warnings.filterwarnings("ignore", category=FutureWarning)

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "project_data.py").is_file()),
    Path.cwd(),
)
PREDICTIONS_DIR = Path(os.environ.get("FAIRNESS_PREDICTIONS_DIR", REPO_ROOT / "results" / "predictions"))
OUTPUT_DIR = Path(os.environ.get("FAIRNESS_OUTPUT_DIR", REPO_ROOT / "results" / "fairness"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Reading predictions from:", PREDICTIONS_DIR)
print("Writing tables to:      ", OUTPUT_DIR)

In [ ]:
# ---- Which runs to analyse -------------------------------------------------
SPLIT_STRATEGY = "stratified"   # keep runs with this split (runs with no split metadata are kept and flagged)
CONFIGURATION = "starter_v1"    # keep runs with this configuration name; None keeps all
FOCUS_RUN = None                # e.g. ("nn", "application_history"); None = highest ROC-AUC run

# ---- Protected / audit attributes ------------------------------------------
PROTECTED_ATTRIBUTES = ["gender", "age_group"]

# ---- Operating points ------------------------------------------------------
FIXED_THRESHOLD = 0.5           # the threshold the model notebooks used
OPERATING_RATE = None           # share flagged at the prevalence-matched point; None = observed default rate
PRIMARY_OP = "prevalence_matched"   # operating point used for plots, trade-off table, mitigation

# ---- Small-group handling --------------------------------------------------
MIN_GROUP_N = 200               # groups smaller than this are shown but excluded from gaps
MIN_DEFAULTS = 30               # ...as are groups with fewer than this many defaulters (or non-defaulters)

# ---- Satisficing tolerances (starting points: justify or change them in the report) ----
AUC_TOLERANCE = 0.005           # a run must be within this of the best ROC-AUC
MAX_FPR_GAP = 0.02              # max-min gap in FPR across groups
MAX_TPR_GAP = 0.10              # max-min gap in TPR across groups
MAX_DENIAL_RATIO = 1.25         # highest / lowest denial rate (adverse-outcome ratio, four-fifths style)

# ---- Uncertainty / mitigation ----------------------------------------------
N_BOOT = 300
BOOTSTRAP_AUC = True            # AUC gaps are the slowest part; set False for a quick run
MITIGATION_TARGET = "selection_rate"   # "selection_rate" or "fpr"
RANDOM_STATE = 42

AGE_ORDER = ["<25", "25-34", "35-44", "45-54", "55-64", "65+", "UNKNOWN"]

## 2. Load predictions and choose runs

Column names in the exported CSVs are matched case-insensitively against the alias lists below. If your export uses different names, add them to `COLUMN_ALIASES` and re-run; the loader prints the columns it found if something is missing. Run metadata (model, feature set, split, configuration) is read from a JSON file next to each CSV when one exists, then from CSV columns, then from the file name.

In [ ]:
COLUMN_ALIASES = {
    "applicant_id": ["applicant_id", "sk_id_curr", "id"],
    "y_true": ["actual_target", "target", "y_true", "actual", "label", "actual_label"],
    "prob": ["predicted_probability", "probability", "y_prob", "prob", "predicted_proba", "pred_prob", "predicted_prob"],
    "gender": ["gender", "code_gender"],
    "age_years": ["age_years", "age"],
    "age_group": ["age_group"],
}
MODEL_PATTERNS = [
    (r"catboost", "catboost"), (r"lightgbm|lgbm", "lightgbm"),
    (r"neural|(?<![a-z])nn(?![a-z])", "nn"), (r"logistic", "logistic_regression"),
]


def resolve_columns(raw, path):
    lowered = {c.lower().strip(): c for c in raw.columns}
    found = {k: next((lowered[a] for a in names if a in lowered), None) for k, names in COLUMN_ALIASES.items()}
    missing = [k for k in ("y_true", "prob", "gender") if found[k] is None]
    if found["age_group"] is None and found["age_years"] is None:
        missing.append("age_group or age_years")
    if missing:
        raise KeyError(f"{path.name}: could not find {missing}. Columns present: {list(raw.columns)}. "
                       "Add the right names to COLUMN_ALIASES.")
    return found


def read_metadata(path):
    for candidate in [path.with_suffix(".json"), path.with_name(path.stem + "_metadata.json"),
                      path.with_name(path.stem + ".metadata.json"), path.with_name(path.stem + "_meta.json")]:
        if candidate.exists():
            return json.loads(candidate.read_text())
    return {}


def meta_get(meta, key):
    if key in meta:
        return meta[key]
    for value in meta.values():
        if isinstance(value, dict) and key in value:
            return value[key]
    return None


def standardise(raw, cols):
    gender = (raw[cols["gender"]].astype("string").str.strip().fillna("UNKNOWN")
              .replace({"XNA": "UNKNOWN", "": "UNKNOWN", "nan": "UNKNOWN"}).astype(str))
    age = pd.to_numeric(raw[cols["age_years"]], errors="coerce") if cols["age_years"] else pd.Series(np.nan, index=raw.index)
    if cols["age_group"]:
        age_group = raw[cols["age_group"]].astype("string").str.strip()
    else:
        age_group = pd.cut(age, bins=[0, 25, 35, 45, 55, 65, np.inf], labels=AGE_ORDER[:-1], right=False).astype("string")
    df = pd.DataFrame({
        "applicant_id": raw[cols["applicant_id"]] if cols["applicant_id"] else raw.index,
        "y": pd.to_numeric(raw[cols["y_true"]], errors="coerce"),
        "prob": pd.to_numeric(raw[cols["prob"]], errors="coerce"),
        "gender": gender, "age_years": age,
        "age_group": age_group.fillna("UNKNOWN").astype(str),
    })
    before = len(df)
    df = df.dropna(subset=["y", "prob"]).reset_index(drop=True)
    df["y"] = df["y"].astype(int)
    return df, before - len(df)


def load_runs(directory):
    records = []
    for path in sorted(directory.glob("*.csv")):
        raw = pd.read_csv(path)
        cols = resolve_columns(raw, path)
        df, dropped = standardise(raw, cols)
        meta = read_metadata(path)
        name = path.stem.lower()

        def pick(key, fallback=None):
            value = meta_get(meta, key)
            if value is None:
                lowered = {c.lower(): c for c in raw.columns}
                if key in lowered:
                    value = raw[lowered[key]].iloc[0]
            return value if value is not None else fallback

        model = pick("model")
        if model is None:
            model = next((label for pattern, label in MODEL_PATTERNS if re.search(pattern, name)), "unknown")
        feature_set = pick("feature_set")
        if feature_set is None:
            feature_set = "application_history" if "history" in name else ("application_only" if "only" in name else "unknown")
        created = pick("created_at_utc") or pick("created") or pd.Timestamp(path.stat().st_mtime, unit="s", tz="UTC").isoformat()
        records.append(dict(model=str(model).lower(), feature_set=str(feature_set),
                            split_strategy=pick("split_strategy", "unknown"), configuration=pick("configuration", "unknown"),
                            run_id=pick("run_id", path.stem), created=str(created), file=path.name,
                            rows=len(df), dropped_rows=dropped, df=df))
    return records


all_records = load_runs(PREDICTIONS_DIR)
if not all_records:
    raise FileNotFoundError(f"No prediction CSVs in {PREDICTIONS_DIR}. Run section 7 of the model notebooks first.")
inventory = pd.DataFrame([{k: v for k, v in r.items() if k != "df"} for r in all_records])
print(f"Found {len(inventory)} prediction file(s):")
display(inventory[["model", "feature_set", "split_strategy", "configuration", "created", "rows", "dropped_rows", "file"]])

In [ ]:
# Keep matching runs, then the latest file per (model, feature set)
keep = [r for r in all_records
        if (SPLIT_STRATEGY is None or r["split_strategy"] in (SPLIT_STRATEGY, "unknown"))
        and (CONFIGURATION is None or r["configuration"] in (CONFIGURATION, "unknown"))]
if not keep:
    raise ValueError("Filters removed every run; check SPLIT_STRATEGY / CONFIGURATION against the inventory above.")

selected = {}
for r in sorted(keep, key=lambda r: r["created"]):
    selected[(r["model"], r["feature_set"])] = r     # later runs overwrite earlier ones


def operating_thresholds(df):
    rate = df["y"].mean() if OPERATING_RATE is None else OPERATING_RATE
    return {"fixed_0.5": FIXED_THRESHOLD,
            "prevalence_matched": float(np.quantile(df["prob"], 1 - rate))}


def fast_auc(y, s):
    y = np.asarray(y)
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    ranks = rankdata(s)
    return (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


for key, rec in selected.items():
    rec["thresholds"] = operating_thresholds(rec["df"])
    rec["auc"] = fast_auc(rec["df"]["y"], rec["df"]["prob"])

# Paired comparison only makes sense if every run scored the same applicants
id_sets = {k: frozenset(r["df"]["applicant_id"]) for k, r in selected.items()}
paired = len(set(id_sets.values())) == 1
print("All selected runs cover the same applicants:", paired)
if not paired:
    print("WARNING: runs use different applicant sets (different splits?). Gaps are not directly comparable across runs.")

if FOCUS_RUN is None:
    focus_key = max(selected, key=lambda k: selected[k]["auc"])
else:
    focus_key = tuple(FOCUS_RUN)
    if focus_key not in selected:
        raise KeyError(f"FOCUS_RUN {focus_key} not among {list(selected)}")
focus = selected[focus_key]
focus_df = focus["df"]
print(f"\nFocus run: {focus_key[0]} / {focus_key[1]}  (validation ROC-AUC {focus['auc']:.4f}, n={len(focus_df):,})")
display(pd.DataFrame([{"model": k[0], "feature_set": k[1], "roc_auc": r["auc"], "file": r["file"],
                       "thr_prevalence_matched": r["thresholds"]["prevalence_matched"]} for k, r in selected.items()]).round(4))

### Metric helpers

Everything below is plain NumPy/pandas, so there is no extra fairness library to install. `group_table` returns one row per group; `gap_summary` turns it into the max-min gaps used for satisficing. Groups below `MIN_GROUP_N` / `MIN_DEFAULTS` are shown but marked `eligible=False` and excluded from the gaps.

In [ ]:
def _div(a, b):
    return a / b if b > 0 else np.nan


def confusion_rates(y, pred):
    y = np.asarray(y); pred = np.asarray(pred)
    tp = int(((y == 1) & (pred == 1)).sum()); fp = int(((y == 0) & (pred == 1)).sum())
    tn = int(((y == 0) & (pred == 0)).sum()); fn = int(((y == 1) & (pred == 0)).sum())
    n = len(y)
    return dict(n=n, defaults=int(y.sum()), base_rate=_div(y.sum(), n), selection_rate=_div(tp + fp, n),
                tpr=_div(tp, tp + fn), fpr=_div(fp, fp + tn), ppv=_div(tp, tp + fp),
                accuracy=_div(tp + tn, n), f1=_div(2 * tp, 2 * tp + fp + fn), tp=tp, fp=fp, tn=tn, fn=fn)


def group_table(df, attr, threshold):
    """Per-group metrics. `threshold` may be a scalar or an array with one cut-off per row."""
    p = df["prob"].to_numpy(); y = df["y"].to_numpy()
    pred = (p >= np.broadcast_to(np.asarray(threshold, dtype=float), p.shape)).astype(int)
    rows = []
    for g, idx in df.groupby(attr, sort=False).indices.items():
        r = confusion_rates(y[idx], pred[idx])
        r.update(group=g, auc=fast_auc(y[idx], p[idx]), mean_prob=p[idx].mean())
        r["calibration_gap"] = r["mean_prob"] - r["base_rate"]
        r["eligible"] = bool(r["n"] >= MIN_GROUP_N and r["defaults"] >= MIN_DEFAULTS and (r["n"] - r["defaults"]) >= MIN_DEFAULTS)
        rows.append(r)
    t = pd.DataFrame(rows).set_index("group")
    t["approval_rate"] = 1 - t["selection_rate"]
    order = [g for g in AGE_ORDER if g in t.index] if attr == "age_group" else t.sort_values("n", ascending=False).index.tolist()
    return t.loc[order + [g for g in t.index if g not in order]]


def gap_summary(t):
    e = t[t["eligible"]]
    nan = dict(n_groups=len(e), selection_rate_gap=np.nan, denial_ratio=np.nan, approval_ratio=np.nan, tpr_gap=np.nan,
               fpr_gap=np.nan, equalized_odds_gap=np.nan, ppv_gap=np.nan, auc_gap=np.nan, calibration_spread=np.nan,
               highest_fpr_group=None, lowest_fpr_group=None)
    if len(e) < 2:
        return nan
    spread = lambda c: e[c].max() - e[c].min()
    den = e["selection_rate"]
    return dict(n_groups=len(e), selection_rate_gap=spread("selection_rate"),
                denial_ratio=den.max() / den.min() if den.min() > 0 else np.nan,
                approval_ratio=e["approval_rate"].min() / e["approval_rate"].max(),
                tpr_gap=spread("tpr"), fpr_gap=spread("fpr"), equalized_odds_gap=max(spread("tpr"), spread("fpr")),
                ppv_gap=spread("ppv"), auc_gap=spread("auc"), calibration_spread=spread("calibration_gap"),
                highest_fpr_group=e["fpr"].idxmax(), lowest_fpr_group=e["fpr"].idxmin())


THR = focus["thresholds"][PRIMARY_OP]
print(f"Primary operating point '{PRIMARY_OP}': threshold {THR:.4f} flags "
      f"{(focus_df['prob'] >= THR).mean():.1%} of applicants (observed default rate {focus_df['y'].mean():.1%}).")

## 3. Group composition and base rates

Fairness criteria cannot all hold at once when groups have different default rates. If base rates differ, a classifier with equal calibration and imperfect accuracy cannot also equalize both error rates (Kleinberg et al.; Chouldechova). Expect the observed default rates below to differ across gender and age; part of the gaps in later sections will reflect that, which is why the notebook tracks several criteria rather than one.

In [ ]:
composition = {}
for attr in PROTECTED_ATTRIBUTES:
    t = group_table(focus_df, attr, THR)
    comp = t[["n", "defaults", "base_rate", "eligible"]].copy()
    comp["share_of_applicants"] = comp["n"] / comp["n"].sum()
    composition[attr] = comp
    print(f"\n{attr}")
    display(comp.round(4))

## 4. Per-group performance (focus model)

Shown at both operating points. The prevalence-matched table is the main one; the fixed-0.5 table reproduces what the model notebooks report. Rows with `eligible = False` are too small to trust and are left out of every gap.

In [ ]:
SHOW = ["n", "defaults", "base_rate", "auc", "selection_rate", "tpr", "fpr", "ppv", "accuracy", "f1",
        "mean_prob", "calibration_gap", "eligible"]
per_group = {}
for op, thr in focus["thresholds"].items():
    for attr in PROTECTED_ATTRIBUTES:
        t = group_table(focus_df, attr, thr)
        per_group[(op, attr)] = t
        flagged = (focus_df["prob"] >= thr).mean()
        print(f"\n[{op}] threshold={thr:.4f}, overall flagged={flagged:.2%} - {attr}")
        display(t[SHOW].round(4))
        if flagged < 0.01:
            print("  NOTE: under 1% of applicants are flagged at this threshold, so rate-based gaps here are not informative.")

## 5. Plots: error rates, ROC curves, calibration

In [ ]:
def bar_panel(ax, t, col, title, overall):
    colors = ["#4477AA" if e else "#BBBBBB" for e in t["eligible"]]
    ax.bar(t.index.astype(str), t[col], color=colors)
    ax.axhline(overall, color="k", ls="--", lw=1)
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.2)


y_all = focus_df["y"].to_numpy(); pred_all = (focus_df["prob"].to_numpy() >= THR).astype(int)
overall = confusion_rates(y_all, pred_all)
fig, axes = plt.subplots(len(PROTECTED_ATTRIBUTES), 3, figsize=(13, 3.8 * len(PROTECTED_ATTRIBUTES)), squeeze=False)
for row, attr in enumerate(PROTECTED_ATTRIBUTES):
    t = per_group[(PRIMARY_OP, attr)]
    bar_panel(axes[row][0], t, "selection_rate", f"{attr}: flagged as default", overall["selection_rate"])
    bar_panel(axes[row][1], t, "fpr", f"{attr}: FPR (creditworthy wrongly flagged)", overall["fpr"])
    bar_panel(axes[row][2], t, "tpr", f"{attr}: TPR (defaulters caught)", overall["tpr"])
fig.suptitle(f"{focus_key[0]} / {focus_key[1]} at {PRIMARY_OP} threshold (dashed = overall; grey = too small)", y=1.0)
fig.tight_layout(); plt.show(); plt.close(fig)

In [ ]:
fig, axes = plt.subplots(1, len(PROTECTED_ATTRIBUTES), figsize=(6 * len(PROTECTED_ATTRIBUTES), 4.6), squeeze=False)
for ax, attr in zip(axes[0], PROTECTED_ATTRIBUTES):
    t = per_group[(PRIMARY_OP, attr)]
    for g in t.index[t["eligible"]]:
        sub = focus_df[focus_df[attr] == g]
        fpr_, tpr_, _ = roc_curve(sub["y"], sub["prob"])
        ax.plot(fpr_, tpr_, label=f"{g} (AUC {t.loc[g, 'auc']:.3f})")
    ax.plot([0, 1], [0, 1], "k:", lw=1)
    ax.set_title(f"ROC by {attr}"); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.legend(fontsize=8); ax.grid(alpha=0.2)
fig.tight_layout(); plt.show(); plt.close(fig)

In [ ]:
# Calibration within groups: does a predicted 10% mean ~10% for everyone?
bins = pd.qcut(focus_df["prob"], 10, duplicates="drop")
fig, axes = plt.subplots(1, len(PROTECTED_ATTRIBUTES), figsize=(6 * len(PROTECTED_ATTRIBUTES), 4.6), squeeze=False)
for ax, attr in zip(axes[0], PROTECTED_ATTRIBUTES):
    t = per_group[(PRIMARY_OP, attr)]
    for g in t.index[t["eligible"]]:
        m = focus_df[focus_df[attr] == g].groupby(bins[focus_df[attr] == g], observed=True).agg(
            predicted=("prob", "mean"), observed=("y", "mean"), n=("y", "size"))
        m = m[m["n"] >= 50]
        ax.plot(m["predicted"], m["observed"], marker="o", ms=3, label=str(g))
    lim = max(focus_df["prob"].quantile(0.995), 0.05)
    ax.plot([0, lim], [0, lim], "k:", lw=1)
    ax.set_title(f"Calibration by {attr} (deciles of score)")
    ax.set_xlabel("Mean predicted default probability"); ax.set_ylabel("Observed default rate")
    ax.legend(fontsize=8); ax.grid(alpha=0.2)
fig.tight_layout(); plt.show(); plt.close(fig)

## 6. Fairness gaps across all runs

Gaps are max-min across eligible groups. `denial_ratio` is the highest group denial rate divided by the lowest (1.0 is parity; the four-fifths idea, borrowed from employment guidance, translates to roughly 1.25 or lower). `approval_ratio` is the traditional lowest/highest approval-rate form; it stays close to 1 whenever most applicants are approved, so the denial ratio is the more sensitive screen. These are screening heuristics, not legal determinations.

In [ ]:
rows = []
for (model, fset), rec in selected.items():
    df = rec["df"]
    for op, thr in rec["thresholds"].items():
        flagged = float((df["prob"] >= thr).mean())
        for attr in PROTECTED_ATTRIBUTES:
            rows.append(dict(model=model, feature_set=fset, operating_point=op, threshold=thr,
                             overall_flagged=flagged, attribute=attr, **gap_summary(group_table(df, attr, thr))))
gaps_all = pd.DataFrame(rows)
gaps_all["note"] = np.where(gaps_all["overall_flagged"] < 0.01, "<1% flagged: rate gaps not informative", "")
cols = ["model", "feature_set", "operating_point", "overall_flagged", "attribute", "n_groups", "selection_rate_gap",
        "denial_ratio", "approval_ratio", "tpr_gap", "fpr_gap", "equalized_odds_gap", "ppv_gap", "auc_gap",
        "calibration_spread", "highest_fpr_group", "note"]
for op in gaps_all["operating_point"].unique():
    print(f"\nOperating point: {op}")
    display(gaps_all[gaps_all["operating_point"] == op][cols].drop(columns="operating_point").round(4).reset_index(drop=True))

## 7. Uncertainty (bootstrap)

Single-run gaps on small groups can be noise. This resamples the focus run's validation rows (with replacement, `N_BOOT` times) and reports a 95% interval for each gap. If an interval comfortably includes zero, treat the gap as unproven rather than absent. It also covers only sampling noise, not training variability (different seeds, different splits).

In [ ]:
BOOT_METRICS = ["selection_rate_gap", "tpr_gap", "fpr_gap", "equalized_odds_gap", "ppv_gap"] + (["auc_gap"] if BOOTSTRAP_AUC else [])


def _nspread(a):
    a = a[~np.isnan(a)]
    return a.max() - a.min() if len(a) > 1 else np.nan


def bootstrap_gaps(df, attr, threshold, n_boot=N_BOOT, seed=RANDOM_STATE):
    base = group_table(df, attr, threshold)
    groups = base.index[base["eligible"]].tolist()
    if len(groups) < 2:
        return None
    k = len(groups)
    codes = df[attr].map({g_: i_ for i_, g_ in enumerate(groups)}).fillna(-1).astype(int).to_numpy()
    y = df["y"].to_numpy(); p = df["prob"].to_numpy(); pred = (p >= threshold).astype(int)
    rng = np.random.default_rng(seed); n = len(df)
    draws = {m: [] for m in BOOT_METRICS}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        c = codes[idx]; keep_ = c >= 0
        c = c[keep_]; yy = y[idx][keep_]; pr = pred[idx][keep_]
        count = lambda w: np.bincount(c, weights=w, minlength=k)
        tp = count(yy * pr); fp = count((1 - yy) * pr); fn = count(yy * (1 - pr)); tn = count((1 - yy) * (1 - pr))
        with np.errstate(divide="ignore", invalid="ignore"):
            sel = (tp + fp) / (tp + fp + fn + tn); tpr = tp / (tp + fn); fpr = fp / (fp + tn); ppv = tp / (tp + fp)
        draws["selection_rate_gap"].append(_nspread(sel)); draws["tpr_gap"].append(_nspread(tpr))
        draws["fpr_gap"].append(_nspread(fpr)); draws["ppv_gap"].append(_nspread(ppv))
        draws["equalized_odds_gap"].append(np.nanmax([_nspread(tpr), _nspread(fpr)]) if not np.isnan([_nspread(tpr), _nspread(fpr)]).all() else np.nan)
        if BOOTSTRAP_AUC:
            sc = p[idx][keep_]
            aucs = np.array([fast_auc(yy[c == gi], sc[c == gi]) for gi in range(k)])
            draws["auc_gap"].append(_nspread(aucs))
    point = gap_summary(base)
    out = []
    for m in BOOT_METRICS:
        d = np.array(draws[m], dtype=float)
        out.append(dict(attribute=attr, metric=m, estimate=point[m],
                        ci_low=np.nanpercentile(d, 2.5), ci_high=np.nanpercentile(d, 97.5)))
    return pd.DataFrame(out)


boot_tables = [bootstrap_gaps(focus_df, a, THR) for a in PROTECTED_ATTRIBUTES]
boot = pd.concat([b for b in boot_tables if b is not None], ignore_index=True)
boot["interval_excludes_zero"] = (boot["ci_low"] > 0)
display(boot.round(4))

## 8. Intersectional view

Disparities can be larger for combinations (for example, young men) than for either attribute alone. Cells that fail the size rules are suppressed (blank), because rates on a few dozen applicants are not reportable.

In [ ]:
inter = focus_df.copy()
inter["intersection"] = inter["gender"] + " | " + inter["age_group"]
t = group_table(inter, "intersection", THR)
t["gender"] = [i.split(" | ")[0] for i in t.index]; t["age_group"] = [i.split(" | ")[1] for i in t.index]
shown = t[t["eligible"]]
for col, title in [("selection_rate", "Flagged as default"), ("fpr", "FPR (creditworthy wrongly flagged)"),
                   ("tpr", "TPR (defaulters caught)"), ("base_rate", "Observed default rate")]:
    pivot = shown.pivot(index="gender", columns="age_group", values=col)
    pivot = pivot[[c for c in AGE_ORDER if c in pivot.columns]]
    print(f"\n{title}")
    display(pivot.round(4))
print(f"\n{(~t['eligible']).sum()} intersection cell(s) suppressed for size.")
inter_table = t.drop(columns=["gender", "age_group"])

## 9. Fairness vs. performance trade-off and shortlist

Following the assignment's model-selection framing: **ROC-AUC is the optimizing metric**, and the fairness gaps are **satisficing** filters. A run is shortlisted when (a) its ROC-AUC is within `AUC_TOLERANCE` of the best run and (b) for every protected attribute the FPR gap, TPR gap, and denial ratio stay within the tolerances. The tolerances are policy choices, so state and defend them in the report. Because they are yours to set, it is worth also looking at how the ranking changes as you loosen or tighten them.

In [ ]:
perf = []
for (model, fset), rec in selected.items():
    df = rec["df"]; thr = rec["thresholds"][PRIMARY_OP]
    m = confusion_rates(df["y"], (df["prob"] >= thr).astype(int))
    perf.append(dict(model=model, feature_set=fset, roc_auc=rec["auc"],
                     avg_precision=average_precision_score(df["y"], df["prob"]),
                     f1=m["f1"], recall=m["tpr"], precision=m["ppv"], accuracy=m["accuracy"]))
perf = pd.DataFrame(perf)

g = gaps_all[gaps_all["operating_point"] == PRIMARY_OP]
wide = g.pivot_table(index=["model", "feature_set"], columns="attribute",
                     values=["fpr_gap", "tpr_gap", "denial_ratio", "equalized_odds_gap"]).reset_index()
wide.columns = ["_".join(c).strip("_") if isinstance(c, tuple) else c for c in wide.columns]
tradeoff = perf.merge(wide, on=["model", "feature_set"])

ok = pd.Series(True, index=tradeoff.index)
for attr in PROTECTED_ATTRIBUTES:
    ok &= tradeoff[f"fpr_gap_{attr}"].fillna(np.inf) <= MAX_FPR_GAP
    ok &= tradeoff[f"tpr_gap_{attr}"].fillna(np.inf) <= MAX_TPR_GAP
    ok &= tradeoff[f"denial_ratio_{attr}"].fillna(np.inf) <= MAX_DENIAL_RATIO
tradeoff["passes_fairness"] = ok
tradeoff["within_auc_tolerance"] = tradeoff["roc_auc"] >= tradeoff["roc_auc"].max() - AUC_TOLERANCE
tradeoff["shortlist"] = tradeoff["passes_fairness"] & tradeoff["within_auc_tolerance"]
display(tradeoff.sort_values("roc_auc", ascending=False).round(4).reset_index(drop=True))

if tradeoff["shortlist"].any():
    best = tradeoff[tradeoff["shortlist"]].sort_values("roc_auc", ascending=False).iloc[0]
    print(f"Highest-AUC run that satisfies the fairness tolerances: {best['model']} / {best['feature_set']} (AUC {best['roc_auc']:.4f})")
else:
    print("No run satisfies all tolerances at this operating point. Either the tolerances are too tight for the base-rate "
          "differences in the data, or mitigation is needed (section 10). Report which criteria fail and by how much.")

fig, ax = plt.subplots(figsize=(7, 4.6))
tradeoff["max_eo_gap"] = tradeoff[[f"equalized_odds_gap_{a}" for a in PROTECTED_ATTRIBUTES]].max(axis=1)
for _, r in tradeoff.iterrows():
    ax.scatter(r["roc_auc"], r["max_eo_gap"], s=70, marker="o" if r["shortlist"] else "x", color="#EE7733" if r["shortlist"] else "#4477AA")
    ax.annotate(f"{r['model']}\n{r['feature_set'].replace('application_', 'app+')}", (r["roc_auc"], r["max_eo_gap"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("Validation ROC-AUC (optimizing metric)"); ax.set_ylabel("Worst equalized-odds gap across attributes")
ax.set_title("Performance vs. fairness gap (circle = shortlisted)"); ax.grid(alpha=0.2)
fig.tight_layout(); plt.show(); plt.close(fig)

## 10. Mitigation experiment: group-specific thresholds

This quantifies what closing a gap costs. Validation rows are split 50/50 (stratified by group and outcome). Thresholds are **learned on half A** and **evaluated on half B**, so the reported effect is not fit to the rows it is scored on. Per-group thresholds equalize either the denial rate or the FPR across eligible groups (`MITIGATION_TARGET`) while keeping the overall flagged share the same.

**Do not treat this as a deployable fix.** Applying different cut-offs by gender or age is explicit disparate treatment, which US credit law (ECOA / Regulation B) is likely to prohibit. It is included to measure the *size of the trade-off*. Mitigation that you could actually defend works earlier: reweighting or constraints during training, removing proxy features, or reviewing which features drive the gap.

In [ ]:
def group_threshold_experiment(df, attr, target=MITIGATION_TARGET, seed=RANDOM_STATE):
    base = group_table(df, attr, FIXED_THRESHOLD)
    eligible = base.index[base["eligible"]]
    rng = np.random.default_rng(seed)
    shuffled = df.iloc[rng.permutation(len(df))]
    pos = shuffled.groupby(shuffled[attr].astype(str) + "|" + shuffled["y"].astype(str)).cumcount().to_numpy()
    A, B = shuffled[pos % 2 == 0].reset_index(drop=True), shuffled[pos % 2 == 1].reset_index(drop=True)

    rate = A["y"].mean() if OPERATING_RATE is None else OPERATING_RATE
    t0 = float(np.quantile(A["prob"], 1 - rate))
    target_fpr = ((A["prob"] >= t0) & (A["y"] == 0)).sum() / (A["y"] == 0).sum()
    thresholds = {}
    for g in eligible:
        sub = A[A[attr] == g]
        if target == "selection_rate":
            thresholds[g] = float(np.quantile(sub["prob"], 1 - rate))
        else:
            thresholds[g] = float(np.quantile(sub.loc[sub["y"] == 0, "prob"], 1 - target_fpr))
    thr_group = B[attr].map(thresholds).fillna(t0).to_numpy()

    out = []
    for label, thr in [("single global threshold", t0), ("group-specific thresholds", thr_group)]:
        pred = (B["prob"].to_numpy() >= thr).astype(int)
        m = confusion_rates(B["y"], pred)
        gs = gap_summary(group_table(B, attr, thr))
        out.append(dict(attribute=attr, scheme=label, overall_flagged=m["selection_rate"], accuracy=m["accuracy"], f1=m["f1"],
                        recall=m["tpr"], precision=m["ppv"], selection_rate_gap=gs["selection_rate_gap"],
                        denial_ratio=gs["denial_ratio"], fpr_gap=gs["fpr_gap"], tpr_gap=gs["tpr_gap"]))
    return pd.DataFrame(out), thresholds


mitigation_tables, mitigation_thresholds = [], {}
for attr in PROTECTED_ATTRIBUTES:
    tbl, thr_map = group_threshold_experiment(focus_df, attr)
    mitigation_tables.append(tbl); mitigation_thresholds[attr] = thr_map
mitigation = pd.concat(mitigation_tables, ignore_index=True)
print(f"Target: equalize {MITIGATION_TARGET}. Evaluated on held-out half B of the validation set. Overall ROC-AUC is unchanged because scores are not modified.")
display(mitigation.round(4))
for attr, m in mitigation_thresholds.items():
    print(attr, {k: round(v, 4) for k, v in m.items()})

## 11. Export and auto-generated findings

In [ ]:
exports = {
    "group_metrics_focus_run.csv": pd.concat(
        [t.assign(operating_point=op, attribute=attr).reset_index() for (op, attr), t in per_group.items()], ignore_index=True),
    "fairness_gaps_all_runs.csv": gaps_all,
    "fairness_vs_performance.csv": tradeoff,
    "bootstrap_gap_intervals.csv": boot,
    "intersectional_focus_run.csv": inter_table.reset_index(),
    "mitigation_group_thresholds.csv": mitigation,
}
for name, frame in exports.items():
    frame.to_csv(OUTPUT_DIR / name, index=False)
print("Saved:", *[f"\n  {OUTPUT_DIR / n}" for n in exports])

print(f"\n=== Draft findings: {focus_key[0]} / {focus_key[1]} at '{PRIMARY_OP}' ===")
for attr in PROTECTED_ATTRIBUTES:
    t = per_group[(PRIMARY_OP, attr)]; e = t[t["eligible"]]; gs = gap_summary(t)
    b = boot[(boot["attribute"] == attr)].set_index("metric")
    if len(e) < 2:
        print(f"- {attr}: fewer than two eligible groups, no gap computed."); continue
    hi, lo = e["fpr"].idxmax(), e["fpr"].idxmin()
    print(f"- {attr}: FPR runs from {e['fpr'].min():.3f} ({lo}) to {e['fpr'].max():.3f} ({hi}); gap {gs['fpr_gap']:.3f} "
          f"(95% CI {b.loc['fpr_gap', 'ci_low']:.3f} to {b.loc['fpr_gap', 'ci_high']:.3f}).")
    print(f"  TPR gap {gs['tpr_gap']:.3f}; denial-rate ratio {gs['denial_ratio']:.2f}; "
          f"observed default rates span {e['base_rate'].min():.3f} to {e['base_rate'].max():.3f}; AUC gap {gs['auc_gap']:.3f}.")
    worst_cal = e["calibration_gap"].abs().idxmax()
    print(f"  Largest calibration offset: {worst_cal} ({e.loc[worst_cal, 'calibration_gap']:+.4f} mean predicted minus observed).")

## 12. Ethical considerations and limitations

**What this analysis can and cannot show**
- Results come from the **development validation set** under the stratified split. Holdout and Kaggle data were not evaluated, and the non-random deployment split required by the assignment is not part of the current pipeline, so these fairness numbers may shift under it.
- Gender is recorded only as F/M (plus a handful of unknowns), and age is binned. Other protected or sensitive characteristics (family status, region, education, occupation) are available in the data but are not audited here. Add them to `PROTECTED_ATTRIBUTES` by exporting them in the prediction files.
- The labels measure *observed* default among applicants the bank previously approved. If past approval decisions were biased, the labels carry that bias (selective labels), and rejected applicants are absent entirely.
- Gender and age are currently **predictors** in the models (the data-preparation notebook keeps them). Removing them ("fairness through unawareness") rarely removes the disparity because other features act as proxies, but a with/without comparison is a worthwhile next experiment. It needs refitting, so it lives outside this notebook.
- All gaps are single-run estimates with sampling uncertainty only (section 7).

**Points to address in the written report** (fill in with your results)
1. Which fairness criteria you tracked and why (FPR gap protects creditworthy applicants; TPR gap and PPV protect the bank's loss exposure; denial ratio is the adverse-impact screen).
2. Whether any criterion conflicts with ROC-AUC, using the trade-off table and the shortlist.
3. Whether observed base-rate differences explain the gaps, and whether using those base rates as a basis for decisions is acceptable.
4. Proxy risk: which features correlate with gender or age, and whether they should stay.
5. What the mitigation experiment cost, and why group-specific thresholds are or are not defensible.
6. Deployment safeguards: human review for borderline scores, adverse-action explanations to applicants, and periodic re-audit as the applicant population drifts.